In [21]:
!pip install pandas scikit-learn logging

In [78]:
import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Generate synthetic data
n_samples = 1000
data = {
    'timestamp': pd.date_range(start='2024-01-01', periods=n_samples, freq='h'),
    'cpu_usage': np.random.normal(50, 10, n_samples),       # CPU usage in percentage
    'memory_usage': np.random.normal(60, 15, n_samples),    # Memory usage in percentage
    'network_latency': np.random.normal(100, 20, n_samples), # Network latency in ms
    'disk_io': np.random.normal(75, 10, n_samples),         # Disk I/O in MB/s
    'error_rate': np.random.choice([0, 1], n_samples, p=[0.95, 0.05])  # 5% error rate
}

# Create DataFrame
df = pd.DataFrame(data)
df = df.drop(['timestamp'], axis=1)
# Display the first few rows of the dataset
print(df.head())
print(df.info())

   cpu_usage  memory_usage  network_latency    disk_io  error_rate
0  54.967142     80.990332        86.496435  55.921924           0
1  48.617357     73.869505        97.109627  66.396150           0
2  56.476885     60.894456        84.151602  70.863945           1
3  65.230299     50.295948        93.840769  93.876877           0
4  47.658466     70.473350        62.127707  80.565531           0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   cpu_usage        1000 non-null   float64
 1   memory_usage     1000 non-null   float64
 2   network_latency  1000 non-null   float64
 3   disk_io          1000 non-null   float64
 4   error_rate       1000 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 39.2 KB
None


In [79]:
from sklearn.ensemble import IsolationForest

# Implement anomaly detection using Isolation Forest
def detect_anomalies(data):
    model = IsolationForest(contamination=0.05, random_state=42)
    model.fit(data)
    anomalies = model.predict(data)
    return anomalies

# Detect anomalies in the dataset 
numeric_data = df.select_dtypes(include=[float, int]) # Only numeric columns 
df['anomaly'] = detect_anomalies(numeric_data)

print(df['anomaly'].value_counts()) # -1 denotes an anomaly

 1    950
-1     50
Name: anomaly, dtype: int64


In [80]:
from scipy.stats import zscore

# Calculate z-scores to identify anomalous values per column in anomalous rows
z_scores = numeric_data.apply(zscore)

# Function to identify anomalous columns for each row
def find_anomalous_columns(row, threshold=3):
    return [col for col in numeric_data.columns if abs(z_scores.loc[row.name, col]) > threshold]

# Apply the function to each anomalous row
df['anomalous_columns'] = df.apply(lambda row: find_anomalous_columns(row) if row['anomaly'] == -1 else [], axis=1)

# Display rows with anomalies and their anomalous columns
print(df[df['anomaly'] == -1][['anomaly', 'anomalous_columns']])

     anomaly              anomalous_columns
37        -1                   [error_rate]
38        -1                   [error_rate]
62        -1                   [error_rate]
132       -1                   [error_rate]
179       -1                   [error_rate]
192       -1                   [error_rate]
208       -1                   [error_rate]
241       -1                   [error_rate]
245       -1                   [error_rate]
251       -1                   [error_rate]
262       -1        [cpu_usage, error_rate]
272       -1                   [error_rate]
285       -1                   [error_rate]
315       -1                   [error_rate]
329       -1                   [error_rate]
330       -1                   [error_rate]
334       -1                   [error_rate]
350       -1                   [error_rate]
354       -1                   [error_rate]
371       -1                   [error_rate]
386       -1                   [error_rate]
387       -1                   [

In [81]:
df['anomalous_columns'] = df.apply(lambda row: ', '.join(find_anomalous_columns(row)) if row['anomaly'] == -1 else '0', axis=1)

In [90]:
df['anomaly']

0      1
1      1
2      1
3      1
4      1
      ..
995    1
996    1
997    1
998    1
999    1
Name: anomaly, Length: 1000, dtype: int64

In [91]:
from sklearn.model_selection import train_test_split
# Split the dataset into features (X) and labels (y)
X = df.drop(['anomaly'], axis=1)
y = df['anomaly']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [92]:
from sklearn.tree import DecisionTreeClassifier

# Train a decision tree for root cause analysis
def root_cause_analysis(X_train, y_train, X_test):
    model = DecisionTreeClassifier()
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    return predictions

predicted_causes = root_cause_analysis(X_train, y_train, X_test)

In [86]:
# Example solution recommendation based on root cause
def recommend_solution(root_cause):
    solutions = {
        "network_error": "Restart the network service.",
        "database_issue": "Check the database connection and restart the service.",
        "high_cpu_usage": "Optimize running processes or allocate more resources."
    }
    return solutions.get(root_cause, "No recommendation available.")

# Recommend a solution based on a detected root cause
solution = recommend_solution("network_error")
print(f"Recommended solution: {solution}")

Recommended solution: Restart the network service.


In [94]:
# Simulate a network error by altering the dataset
df.loc[0, 'network_latency'] = 1000  # Simulating high network latency
df

,cpu_usage,memory_usage,network_latency,disk_io,error_rate,anomaly
0,54.967142,80.990332,1000.000000,55.921924,0,1
1,48.617357,73.869505,97.109627,66.396150,0,1
2,56.476885,60.894456,84.151602,70.863945,1,1
3,65.230299,50.295948,93.840769,93.876877,0,1
4,47.658466,70.473350,62.127707,80.565531,0,1
...,...,...,...,...,...,...
995,47.188997,76.052254,101.549610,75.284576,0,1
996,67.976865,59.602181,105.155051,54.221882,0,1
997,56.408429,46.771880,75.164788,71.797022,0,1
998,44.288210,57.553996,106.683528,91.433782,0,1


In [93]:
# Run the troubleshooting agent
anomalies = detect_anomalies(df)
predicted_causes = root_cause_analysis(X_train, y_train, df)
solution = recommend_solution(predicted_causes[0])
print(f"Detected issue: {predicted_causes[0]}")
print(f"Recommended solution: {solution}")

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- anomaly
